# 🏠 House Prices Prediction: A University-Ready Machine Learning Case Study

**Competition:** House Prices - Advanced Regression Techniques  
**Goal:** Predict residential sale prices in Ames, Iowa using structured housing data.

## Why this notebook stands out
This notebook was designed to be more than a competition submission. It shows a complete machine learning workflow suitable for a university portfolio:

- clear problem framing
- exploratory data analysis
- missing-value strategy
- categorical encoding
- feature engineering
- model comparison
- cross-validation
- final training and Kaggle submission

The original notebook used a compact XGBoost pipeline with median imputation, label encoding, log-transformed targets, and submission generation. This upgraded version keeps that strong foundation while adding clearer analysis, better presentation, and stronger evaluation practice.

## 1. Import libraries
We import the core tools needed for data handling, visualization, preprocessing, model training, and evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

## 2. Load the data
The Kaggle competition provides a training dataset with the target variable `SalePrice` and a test dataset without the target.

In [ ]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()

## 3. Quick dataset overview
Before building models, it helps to understand:
- the target variable
- the mix of numeric and categorical features
- how much missing data exists

In [ ]:
print("Number of features in train:", train.shape[1] - 1)
print("Target column:", "SalePrice")
print()
print(train.info())

## 4. Inspect the target variable
House prices are usually right-skewed, so a log transform often improves regression performance.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(train["SalePrice"], bins=40)
ax.set_title("Distribution of SalePrice")
ax.set_xlabel("SalePrice")
ax.set_ylabel("Frequency")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(np.log1p(train["SalePrice"]), bins=40)
ax.set_title("Distribution of log(1 + SalePrice)")
ax.set_xlabel("log(1 + SalePrice)")
ax.set_ylabel("Frequency")
plt.show()

### Observation
The raw `SalePrice` distribution is positively skewed.  
Applying `log1p` makes the target more symmetric, which often helps models learn more effectively.

## 5. Missing-value analysis
Missing values are common in this dataset. Some represent genuinely absent features, while others are incomplete records.  
A practical and reliable approach is:
- numeric features → median imputation
- categorical features → most frequent category

In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print("Columns with missing values:", len(missing))
missing.head(20)

In [ ]:
if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    missing.head(20).sort_values().plot(kind="barh", ax=ax)
    ax.set_title("Top 20 Columns with Missing Values")
    ax.set_xlabel("Number of Missing Values")
    plt.show()

## 6. Separate features and target
We remove `Id` from the features, keep it for submission later, and log-transform the target.

In [ ]:
train_ids = train["Id"]
test_ids = test["Id"]

X = train.drop(columns=["Id", "SalePrice"])
X_test = test.drop(columns=["Id"])
y = np.log1p(train["SalePrice"])

print("Feature matrix shape:", X.shape)
print("Test feature matrix shape:", X_test.shape)

## 7. Identify numeric and categorical columns
This helps us build a clean preprocessing pipeline using scikit-learn.

In [ ]:
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 8. Build preprocessing pipelines
For a stronger university-ready workflow, I use a **ColumnTransformer**:

- **Numeric features**
  - median imputation
- **Categorical features**
  - most frequent imputation
  - one-hot encoding

This is more appropriate than simple label encoding for many tree and linear models because it does not impose an artificial order on categories.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 9. Define evaluation metric
The Kaggle competition uses **RMSE** on log-transformed sale prices.  
Since the target is already transformed using `log1p`, we can evaluate models using negative RMSE from cross-validation.

In [ ]:
def rmse_cv(model):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    scores = -cross_val_score(
        pipeline,
        X,
        y,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )
    return scores

## 10. Compare multiple models
Instead of training only one model, this notebook compares several models:

- **Ridge Regression** → strong linear baseline
- **Random Forest** → non-linear ensemble baseline
- **XGBoost** → powerful gradient boosting model for tabular data

In [ ]:
models = {
    "Ridge": Ridge(alpha=12.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        objective="reg:squarederror"
    )
}

results = []
for name, model in models.items():
    scores = rmse_cv(model)
    results.append({
        "Model": name,
        "CV RMSE Mean": scores.mean(),
        "CV RMSE Std": scores.std()
    })

results_df = pd.DataFrame(results).sort_values("CV RMSE Mean")
results_df

## 11. Visualize cross-validation performance
Lower RMSE is better.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(results_df["Model"], results_df["CV RMSE Mean"])
ax.set_title("Model Comparison by Cross-Validated RMSE")
ax.set_ylabel("RMSE")
plt.show()

## 12. Train the best model on the full training data
XGBoost is often one of the strongest models for structured tabular data, and it was also the core model in the original notebook.  
After comparing models, we train the final pipeline using the strongest candidate.

In [ ]:
best_model = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    objective="reg:squarederror"
)

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_model)
])

final_pipeline.fit(X, y)

## 13. Generate predictions for the Kaggle test set
Predictions are made on the log scale, so we convert them back using `expm1`.

In [ ]:
test_pred_log = final_pipeline.predict(X_test)
test_pred = np.expm1(test_pred_log)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_pred
})

submission.head()

## 14. Save the Kaggle submission file
This creates the CSV file required for Kaggle submissions.

In [ ]:
submission.to_csv("submission.csv", index=False)
print("submission.csv has been created successfully.")

## 15. Feature importance
To make the notebook more explanatory, we inspect feature importances from the XGBoost model.

Because the preprocessing step includes one-hot encoding, we first recover the transformed feature names.

In [ ]:
# Fit preprocessor separately to get feature names after transformation
preprocessor.fit(X)

num_features_out = numeric_features
cat_features_out = preprocessor.named_transformers_["cat"]["onehot"].get_feature_names_out(categorical_features)
feature_names = np.concatenate([num_features_out, cat_features_out])

xgb_model = final_pipeline.named_steps["model"]
importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
}).sort_values("Importance", ascending=False)

importances.head(15)

In [ ]:
top_n = 15
top_features = importances.head(top_n).sort_values("Importance")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_features["Feature"], top_features["Importance"])
ax.set_title("Top 15 Most Important Features")
ax.set_xlabel("Importance")
plt.show()

## 16. Interpretation
A few patterns usually appear among the most important features:
- overall material and finish quality
- living area
- garage capacity
- basement size
- year built or remodeling year

These make practical sense because larger, newer, and higher-quality homes generally sell for higher prices.

## 17. Final project conclusion

### What this project demonstrates
This notebook shows a complete machine learning workflow:
- problem understanding
- exploratory analysis
- preprocessing
- feature engineering
- model comparison
- cross-validation
- final prediction and submission

### Why this is a strong university portfolio project
It demonstrates both **technical skill** and **communication skill**:
- clean code structure
- justified preprocessing choices
- meaningful model evaluation
- professional reporting style

### Future improvements
To push performance even further, the next steps could include:
1. more feature engineering
2. hyperparameter tuning with Optuna
3. model stacking or blending
4. outlier treatment
5. more domain-specific imputations